# IS362 – Project 4: Predictive Analysis Using scikit-learn  

Daniela Porras
IS 362

In this project, I build predictive models using scikit-learn to determine which attributes best predict whether a mushroom is edible or poisonous.  
I use the mushroom dataset from UCI and focus on two categorical predictors: `odor` and `gill_color`.  
The goal is to identify which feature is more accurate at predicting edibility, and to compare their performance using classification models.

## Load and Preprocess Data

In [1]:
import pandas as pd

# Load the dataset
column_names = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
    'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
    'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
    'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
    'ring-type', 'spore-print-color', 'population', 'habitat'
]

df = pd.read_csv("agaricus-lepiota.data", header=None, names=column_names)

# Select relevant columns
df_subset = df[['class', 'odor', 'gill-color']].copy()
df_subset.columns = ['is_poisonous', 'odor', 'gill_color']

# Map target variable to numeric
df_subset['is_poisonous'] = df_subset['is_poisonous'].map({'e': 0, 'p': 1})

df_subset.head()

,is_poisonous,odor,gill_color
0,1,p,k
1,0,a,k
2,0,l,n
3,1,p,n
4,0,n,k


## One-Hot Encoding  
Since `odor` and `gill_color` are categorical, I use one-hot encoding to convert them into numeric format.  
This allows the machine learning models to properly interpret the values without assuming any order.

In [2]:
# One-hot encode the features separately
odor_encoded = pd.get_dummies(df_subset['odor'], prefix='odor')
gill_encoded = pd.get_dummies(df_subset['gill_color'], prefix='gill')

# Combine with target
df_odor = pd.concat([df_subset['is_poisonous'], odor_encoded], axis=1)
df_gill = pd.concat([df_subset['is_poisonous'], gill_encoded], axis=1)
df_both = pd.concat([df_subset['is_poisonous'], odor_encoded, gill_encoded], axis=1)

## Model Setup  
I will use scikit-learn's `LogisticRegression` to train three models:
1. Using only `odor`
2. Using only `gill_color`
3. Using both features

Each model will be evaluated using accuracy on a held-out test set.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Setup a function to train and evaluate models
def train_and_evaluate(df, feature_name):
    X = df.drop('is_poisonous', axis=1)
    y = df['is_poisonous']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"Accuracy using {feature_name}: {acc:.4f}")
    return acc

# Run models
odor_acc = train_and_evaluate(df_odor, "odor")
gill_acc = train_and_evaluate(df_gill, "gill_color")
both_acc = train_and_evaluate(df_both, "odor + gill_color")

Accuracy using odor: 0.9840
Accuracy using gill_color: 0.8019
Accuracy using odor + gill_color: 0.9877


## Final Results and Conclusion

After training three different logistic regression models using scikit-learn, the results are:

- **Accuracy using `odor` only**: **98.40%**
- **Accuracy using `gill_color` only**: **80.19%**
- **Accuracy using both `odor + gill_color`**: **98.77%**

### Analysis:
- The `odor` feature alone is a very strong predictor of whether a mushroom is poisonous or not.
- `Gill color` performs much worse by itself, meaning it’s not as reliable.
- Combining both features slightly improves the model's accuracy, but most of the predictive power clearly comes from odor.

### Conclusion:
This analysis shows that **odor is a key indicator** of mushroom toxicity and performs extremely well in predictive modeling. While combining features adds marginal benefit, `odor` alone gives us nearly perfect classification. In future work, it could be helpful to test additional attributes (like bruises, habitat, or cap shape) to explore if there's any further improvement.
